# Signal Recovery with Gaussian Process

## 1- Signal Recovery With Perfect Classifiers:
- Segment the df_parti_hr.
- Iterate over the segmented df_parti_hr until a segment with motion artifact is found.
- Get ~30 'good quality' segments before/after the motion artifact based on the `labels` (perfect classifier).
- Train the MTGP with those segments.
- Estimate the new heart rate values, at the timestamps of the reference signals
- Calculate the error between the reference signal and cushion signal

In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib qt

from ma.utils import load_participant_data, put_metadata_to_segments
from ma.gp_utils import ecg_r_peak_detection, calculate_hr_or_rr, combine_ma_intervals
from ma.gp_visu import plot_signals_with_r_peaks, plot_signals_with_hr
import matplotlib.pyplot as plt
plt.rcParams['pgf.preamble'] = r'\usepackage{tikz}'
from matplotlib.figure import Figure
from ma.gp import GPPerfectClassifier, GPModelClassifier
import pandas as pd
import numpy as np

PARTI_NO = 17
SIGNAL_NAMES = ["ecg1"]
CLASS_NAME = "HR"

In [2]:
gp_pc = GPPerfectClassifier(
    parti_no=PARTI_NO, signal_names=SIGNAL_NAMES, class_name=CLASS_NAME
)
gp_pc.recover_signals()

 c:\Users\firat\AppData\Local\Programs\Python\Python312\Lib\site-packages\GPy\util\normalizer.py:94: UserWarning:Some values of Y have standard deviation of zero. Resetting to 1.0 to avoid divide by zero errors.


In [37]:
import GPy

In [ ]:
df_train_left = gp_pc.df_parti_hr_rr["ecg1"].loc[1238:1549.5]
df_train_left = df_train_left[df_train_left["hr"] > 59]
df_ma = gp_pc.df_parti_hr_rr["ecg1"].loc[1549:1593] 
df_ref

time_hr_left = df_train_left.index.to_numpy()[:, None]
hr_data_left = df_train_left["hr"].to_numpy()[:, None]

time_ma = df_ma.index.to_numpy()[:, None]
hr_ma = df_ma["hr"].to_numpy()[:, None]

fig, ax = plt.subplots()
ax.plot(time_hr_left, hr_data_left, color = "green", linewidth = 3.0, marker = "o", label = "HR - Train")
ax.plot(time_ma, hr_ma, color = "red", marker = "o", label = "HR - MA")
ax.grid()
ax.legend()

In [62]:
df_train_left

,hr,sq_labels,ma_labels
1238.5700,65.075922,0,0
1239.4960,64.516129,0,0
1240.4220,65.075922,0,0
1241.3360,66.225166,0,0
1242.2695,62.434964,0,0
...,...,...,...
1545.9765,73.800738,0,0
1546.7775,76.045627,0,0
1547.5625,76.824584,0,0
1548.3280,80.000000,0,0


In [60]:
rbf_kernel = GPy.kern.RBF(input_dim=1, variance=10.0, lengthscale=5.0)
periodic_kernel_respiration = GPy.kern.PeriodicExponential(input_dim=1, variance=10.0, lengthscale=1.0, period=1.0)
periodic_kernel_mayer = GPy.kern.PeriodicExponential(input_dim=1, variance=10.0, lengthscale=1.0, period=10.0)

# Combine kernels
combined_kernel = rbf_kernel + periodic_kernel_respiration + periodic_kernel_mayer

gp_model = GPy.models.GPRegression(time_hr_left, hr_data_left, kernel=combined_kernel, normalizer=True)
gp_model.optimize()

In [61]:
Y_pred_hr, Y_var_hr = gp_model.predict(time_ma)

# Plot results
# Heart Rate predictions
ax.plot(time_ma, Y_pred_hr, 'bx-', label='Predicted HR')
ax.fill_between(
    time_ma.flatten(),
    Y_pred_hr.flatten() - 1.96 * np.sqrt(Y_var_hr.flatten()),
    Y_pred_hr.flatten() + 1.96 * np.sqrt(Y_var_hr.flatten()),
    color="blue",
    alpha=0.2,
)

In [3]:
_ = gp_pc.plot_recovered_signals(model_name="gpy", signal_name=SIGNAL_NAMES[0], mark_train_signals=False)

In [10]:
gp_pc.calculate_ma_error()
display(gp_pc.get_ma_errors(signal_name=SIGNAL_NAMES[0], model_name="gpy"))
# display(gp_pc.get_ma_errors(signal_name=SIGNAL_NAMES[0], model_name="gp_combined_se"))
# gp_pc.get_ma_errors(signal_name=SIGNAL_NAMES[0], model_name="gp_combined_matern")

,errors_raw,errors_recovered
mse,478.858870,69.830263
mae,13.646626,5.833814
rmse,21.882844,8.356450
med_abs_err,0.743172,3.606738


In [ ]:
gp_pc.calculate_overall_error()
display(gp_pc.get_overall_errors(signal_name=SIGNAL_NAMES[0], model_name="gp_rational_quadratic"))
display(gp_pc.get_overall_errors(signal_name=SIGNAL_NAMES[0], model_name="gp_combined_se"))
gp_pc.get_overall_errors(signal_name=SIGNAL_NAMES[0], model_name="gp_combined_matern")

## 1- Signal Recovery With Non-Perfect Classifiers:
- Segment the df_parti_hr.
- Predict the `signal quality` & `motion artifacts` by using pre-trained models (non-perfect classifier).
- Iterate over the segmented df_parti_hr until a segment with motion artifact is found.
- Get ~30 'good quality' segments before/after the motion artifact.
- Train the MTGP with those segments.
- Estimate the new heart rate values, at the timestamps of the reference signals
- Calculate the error between the reference signal and cushion signal

In [ ]:
gp_mc = GPModelClassifier(
    parti_no=PARTI_NO, signal_names=SIGNAL_NAMES, class_name=CLASS_NAME
)
gp_mc.recover_signals()

In [ ]:
_ = gp_mc.plot_recovered_signals(model_name="gp_combined_se", signal_name=SIGNAL_NAMES[0], mark_train_signals=False)

In [ ]:
gp_mc.calculate_ma_error()
display(gp_mc.get_ma_errors(signal_name=SIGNAL_NAMES[0], model_name="gp_rational_quadratic"))
display(gp_mc.get_ma_errors(signal_name=SIGNAL_NAMES[0], model_name="gp_combined_se"))
gp_mc.get_ma_errors(signal_name=SIGNAL_NAMES[0], model_name="gp_combined_matern")

In [ ]:
gp_mc.calculate_overall_error()
display(gp_mc.get_overall_errors(signal_name=SIGNAL_NAMES[0], model_name="gp_rational_quadratic"))
display(gp_mc.get_overall_errors(signal_name=SIGNAL_NAMES[0], model_name="gp_combined_se"))
gp_mc.get_overall_errors(signal_name=SIGNAL_NAMES[0], model_name="gp_combined_matern")

--------

# Manual Steps:

In [ ]:
PARTI_NO = 17
SIGNAL_NAMES = ["scg1x"]
CLASS_NAME = "RR"
REF_SIGNAL = "resp_ref" # ecg_ref or resp_ref
# Not-Segmented data:
df_parti = load_participant_data(participant_number=PARTI_NO, signal_names=SIGNAL_NAMES, class_name=CLASS_NAME, bandpassed=True, time_offset=True)
# Segmented data
parti_data, parti_labels= load_participant_data(
    participant_number=PARTI_NO,
    signal_names=SIGNAL_NAMES,
    bandpassed=True,
    normalized=False,
    segmented=True,
    class_name=CLASS_NAME,
    load_labels=True,
    time_offset=True,
)
df_parti_segmented = put_metadata_to_segments(segments=parti_data, labels=parti_labels, parti_no=PARTI_NO)
# Reference Data
df_ref_segmented = load_participant_data(participant_number=PARTI_NO, signal_names=[REF_SIGNAL], class_name=CLASS_NAME, segmented=True, time_offset=True)

# Manual Steps:


# Peak Detection

In [ ]:
df_ref = load_participant_data(participant_number=PARTI_NO, signal_names=[REF_SIGNAL], class_name=CLASS_NAME, bandpassed=True, time_offset=True)

In [ ]:
from ma.gp_utils import scg_peak_detection
r_peaks_parti = scg_peak_detection(df_segmented=df_parti_segmented, signal_name=SIGNAL_NAMES[0])
r_peaks_ref = scg_peak_detection(df_segmented=df_ref_segmented, signal_name=REF_SIGNAL, distance=20, window_size=10, dynamic_height_multiplier=0.1, prominence=0.1)
fig_signal_with_r_peak = plot_signals_with_r_peaks(df_parti_segmented=df_parti_segmented, df_ref=df_ref, df_parti=df_parti, peaks_parti=r_peaks_parti, peaks_ref=r_peaks_ref, ref_signal=REF_SIGNAL)

# Heart Rate Calculation

In [ ]:
df_parti_hr = calculate_hr_or_rr(df_parti_segmented=df_parti_segmented, peaks=r_peaks_parti, class_name=CLASS_NAME)
df_ref_hr = calculate_hr_or_rr(df_parti_segmented=df_ref_segmented, peaks=r_peaks_ref, class_name=CLASS_NAME)

In [ ]:
fig = plot_signals_with_hr(
    df_parti_segmented=df_parti_segmented,
    df_parti_hr=df_parti_hr,
    df_ref=df_ref,
    df_ref_hr=df_ref_hr,
    df_parti=df_parti,
    peaks_parti=r_peaks_parti,
    peaks_ref=r_peaks_ref,
    class_name=CLASS_NAME
)

# Finding the Train & Test Data: 

In [ ]:
ma_intervals = []
r_peaks_flattened = [
    value + len(df_ref_segmented[0]) * idx for idx, sublist in enumerate(r_peaks_parti) for value in sublist
]

r_peak_times = df_parti.index[r_peaks_flattened]
for segment_no, segment in enumerate(df_parti_segmented):
    ax = fig.get_axes()[-1]
    if segment_no == 0:
        # Special Case
        continue

    if segment.attrs[f"{SIGNAL_NAMES[0]}_ma"] == 1:
        start_idx = np.where((r_peak_times - segment.index[0]) < 0)[0][-1]
        stop_idx = np.where((r_peak_times - segment.index[-1]) > 0)[0][0]

        ma_intervals.append([start_idx - 1, stop_idx])
        
        if stop_idx >= len(df_parti_hr):
            # Special Case:
            break

        ax.axvspan(
            df_parti_hr.index[start_idx - 1],
            df_parti_hr.index[stop_idx],
            color="lightgray",
            alpha=0.4,
        )
fig

-----------